[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_09_Hyperparameter_Tuning/01_hyperparameter_tuning.ipynb)

# Episode 22 – Hyperparameter Tuning

**Machine Learning Bootcamp** | Module 09

---

## 🎯 Learning Objectives
- Distinguish hyperparameters from model parameters
- Apply Grid Search and Random Search with cross-validation
- Visualise and interpret hyperparameter search results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score

sns.set_theme(style='whitegrid')

## 1. Parameters vs Hyperparameters

| Type | Example | Learned from data? |
|------|---------|--------------------|
| **Parameter** | Weights in a neural network, coefficients in linear regression | ✅ Yes |
| **Hyperparameter** | Learning rate, number of trees, max depth, C in SVM | ❌ No – set before training |

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 2. Grid Search

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print(f'Best CV F1: {grid_search.best_score_:.4f}')
print(f'Test F1:    {accuracy_score(y_test, grid_search.predict(X_test)):.4f}')

In [ ]:
# Visualise Grid Search results (n_estimators vs max_depth for best min_samples_split)
results = pd.DataFrame(grid_search.cv_results_)
pivot = results[results['param_min_samples_split'] == grid_search.best_params_['min_samples_split']].pivot_table(
    index='param_max_depth', columns='param_n_estimators', values='mean_test_score'
)
plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Grid Search CV F1 Scores')
plt.tight_layout(); plt.show()

## 3. Random Search

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators':      randint(50, 500),
    'max_depth':         [None, 3, 5, 7, 10, 15],
    'min_samples_split': randint(2, 20),
    'max_features':      ['sqrt', 'log2', None],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=40, cv=5, scoring='f1',
    random_state=42, n_jobs=-1, verbose=0
)
random_search.fit(X_train, y_train)

print('Best params:', random_search.best_params_)
print(f'Best CV F1: {random_search.best_score_:.4f}')
print(f'Test Acc:   {accuracy_score(y_test, random_search.predict(X_test)):.4f}')

## 🏋️ Exercises

1. Run `GridSearchCV` on an SVM with `C` ∈ [0.1, 1, 10, 100] and `gamma` ∈ ['scale', 0.01, 0.1]. Compare with Random Forest.
2. Add `return_train_score=True` to `GridSearchCV` and plot train vs. validation score per configuration.
3. Try `BayesSearchCV` from the `scikit-optimize` library on the same search space. Does it find a better solution in fewer evaluations?

---
**Next ▶ [Episode 23 – ML Pipelines](02_pipelines.ipynb)**